In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [4]:
df=pd.read_csv("fake_bills.csv",sep=';')
df

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length
0,True,171.81,104.86,104.95,4.52,2.89,112.83
1,True,171.46,103.36,103.66,3.77,2.99,113.09
2,True,172.69,104.48,103.50,4.40,2.94,113.16
3,True,171.36,103.91,103.94,3.62,3.01,113.51
4,True,171.73,104.28,103.46,4.04,3.48,112.54
...,...,...,...,...,...,...,...
1495,False,171.75,104.38,104.17,4.42,3.09,111.28
1496,False,172.19,104.63,104.44,5.27,3.37,110.97
1497,False,171.80,104.01,104.12,5.51,3.36,111.95
1498,False,172.06,104.28,104.06,5.17,3.46,112.25


In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
df.head()

,is_genuine,diagonal,height_left,height_right,margin_low,margin_up,length
0,True,171.81,104.86,104.95,4.52,2.89,112.83
1,True,171.46,103.36,103.66,3.77,2.99,113.09
2,True,172.69,104.48,103.50,4.40,2.94,113.16
3,True,171.36,103.91,103.94,3.62,3.01,113.51
4,True,171.73,104.28,103.46,4.04,3.48,112.54


In [8]:
import pandas as pd

# ---------- 1. Load data ----------
# The dataset uses ';' as the separator
df = pd.read_csv("fake_bills.csv", sep=";")

print("=" * 40)
print("BEFORE CLEANING")
print("=" * 40)
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# ---------- 2. Standardize column names ----------
df.columns = df.columns.str.strip().str.lower()

# ---------- 3. Remove duplicates ----------
df = df.drop_duplicates()

# ---------- 4. Handle missing values ----------
# margin_low has some missing values, so fill with the column median
# (median is safer than mean because it is not affected by outliers)
feature_cols = [c for c in df.columns if c != "is_genuine"]
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

# Drop any row where the target itself is missing
df = df.dropna(subset=["is_genuine"])

# ---------- 5. Convert target to 1/0 ----------
df["is_genuine"] = df["is_genuine"].astype(int)

# ---------- 6. Outlier handling (IQR capping) ----------
# Extreme values are capped, not deleted, so no rows are lost.
# Set to False if you want to skip this step.
CAP_OUTLIERS = True

if CAP_OUTLIERS:
    for col in feature_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df[col] = df[col].clip(lower=lower, upper=upper)

# ---------- 7. Final checks ----------
assert df.isnull().sum().sum() == 0, "Missing values still present!"
assert set(df["is_genuine"].unique()) <= {0, 1}, "Target must be 0 or 1!"

print("\n" + "=" * 40)
print("AFTER CLEANING")
print("=" * 40)
print("Shape:", df.shape)
print("Missing values left:", df.isnull().sum().sum())
print("\nClass balance:\n", df["is_genuine"].value_counts())
print("\nSummary statistics:\n", df.describe().round(3))

# ---------- 8. Save ----------
df.to_csv("clean_bills.csv", index=False)
print("\nSaved: clean_bills.csv")

BEFORE CLEANING
Shape: (1500, 7)

Column types:
 is_genuine         bool
diagonal        float64
height_left     float64
height_right    float64
margin_low      float64
margin_up       float64
length          float64
dtype: object

Missing values:
 is_genuine       0
diagonal         0
height_left      0
height_right     0
margin_low      37
margin_up        0
length           0
dtype: int64

Duplicate rows: 0

AFTER CLEANING
Shape: (1500, 7)
Missing values left: 0

Class balance:
 is_genuine
1    1000
0     500
Name: count, dtype: int64

Summary statistics:
        is_genuine  diagonal  height_left  height_right  margin_low  margin_up  \
count    1500.000  1500.000     1500.000      1500.000    1500.000   1500.000   
mean        0.667   171.958      104.030       103.920       4.477      3.152   
std         0.472     0.304        0.299         0.324       0.643      0.231   
min         0.000   171.120      103.205       103.050       2.980      2.510   
25%         0.000   171.750  

In [6]:
df.shape

(1500, 7)

In [7]:
X = df.drop("is_genuine", axis=1)
y = df["is_genuine"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)